# Modelagem supervisionada: pré-processamento e baseline (Fase 3)

**Objetivo:** montar o pipeline de pré-processamento e treinar os modelos de referência
(baselines) para prever se um aluno **não** será alfabetizado (classe de risco).

**Entradas:**
- `data/sample/features_aluno`: a camada Gold (1 linha por aluno avaliado em 2024).
- `src/config/features_modelo.json`: as features e o tratamento de cada uma, definidos
  no notebook `02_selecao_features`. Nenhuma lista de features é digitada aqui.

## 1. Setup e carga


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42   # semente fixa: garante que os sorteios se repitam em qualquer execução

CAMINHO_GOLD = "../data/sample/features_aluno"
CAMINHO_DECISOES = Path("../src/config/features_modelo.json")

df = pd.read_parquet(CAMINHO_GOLD)
decisoes = json.loads(CAMINHO_DECISOES.read_text(encoding="utf-8"))

NUMERICAS_LOG = decisoes["numericas_log"]
NUMERICAS_PADRAO = decisoes["numericas_padrao"]
CATEGORICAS = decisoes["categoricas"]
BINARIAS = decisoes["binarias"]
FEATURES = NUMERICAS_LOG + NUMERICAS_PADRAO + CATEGORICAS + BINARIAS

print(f"Gold: {df.shape} | features lidas do JSON: {len(FEATURES)}")

Gold: (1610754, 35) | features lidas do JSON: 24


## 2. Features (X), alvo (y) e grupos

- **X:** as 24 features. As binárias são convertidas de verdadeiro/falso/vazio para
  1/0/vazio, porque o `SimpleImputer` não aceita valores lógicos com vazios
  (seção 8 do notebook 02).
- **y:** o alvo invertido, **1 = não alfabetizado (risco)** (seção 3 do notebook 02).
- **grupos:** o município de cada aluno, usado para que treino e teste nunca
  compartilhem o mesmo município.

In [2]:
X = df[FEATURES].copy()
for coluna in BINARIAS:
    X[coluna] = X[coluna].astype(float)   # True/False/vazio -> 1.0/0.0/NaN

y = (~df[decisoes["alvo"]]).astype(int).rename("nao_alfabetizado")
grupos = df[decisoes["coluna_grupo"]]

print(f"X: {X.shape}")
print(f"Taxa de risco (y = 1): {y.mean():.1%}")
print(f"Municípios: {grupos.nunique()}")

X: (1610754, 24)
Taxa de risco (y = 1): 40.6%
Municípios: 5450
